# CardioAssist AI - Fine-tuning


In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# !pip uninstall -y transformers trl unsloth
!pip install -U unsloth

In [ ]:
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes
!pip install transformers datasets

In [ ]:
import transformers
import trl
import unsloth
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("Unsloth:", unsloth.__version__)

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
import json
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# Face: https://huggingface.co/datasets/AKCIT/MedPT
HF_DATASET_NAME = "AKCIT/MedPT"

SPECIALTY_FILTER = "cardiologista"

MINIMUM_ANSWER_LENGTH = 40

VALIDATION_FRACTION = 0.1
SPLIT_SEED = 3407

EVALUATION_STEPS = 1

COMPARISON_SAMPLE_SIZE = 5
COMPARISON_MAX_NEW_TOKENS = 256

OUTPUT_PATH_DATASET = "/content/drive/MyDrive/FIAP/ChallengeFase3/dataset_cardiologiaV2.json"
max_seq_length = 1024
dtype = None
load_in_4bit = True
fourbit_models = [
    "unsloth/mistral-7b-v0.3-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/llama-3-8b-bnb-4bit",
    "unsloth/llama-3-8b-Instruct-bnb-4bit",
    "unsloth/llama-3-70b-bnb-4bit",
    "unsloth/Phi-3-mini-4k-instruct",
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit",
]


In [ ]:
# 1. Download do dataset direto do Hugging Face. 
data_bruto = medpt.to_pandas()

print("Linhas no MedPT completo:", len(data_bruto))
print("Colunas:", list(data_bruto.columns))
data_bruto.head()

In [ ]:
# 2. Seleção das perguntas de cardiologia.
especialidades = data_bruto["medical_specialty"].fillna("").str.lower()
data_cardio = data_bruto[especialidades.str.contains(SPECIALTY_FILTER)].copy()

print("Linhas de cardiologia:", len(data_cardio))

In [ ]:
# 3. Anonimização. As perguntas do MedPT são texto livre escrito por pacientes,
# então podem trazer nome, telefone, e-mail, CPF e datas. Os padrões abaixo
# trocam esses dados por marcadores, sem apagar valores clínicos como
# "190/125 mmHg" ou "88 bpm".
import re

ANONYMIZATION_RULES = [
    (re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"), "[EMAIL]"),
    (re.compile(r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b"), "[CPF]"),
    (re.compile(r"\(?\b\d{2}\)?\s?9?\d{4}[-\s]?\d{4}\b"), "[TELEFONE]"),
    (re.compile(r"\b\d{1,2}/\d{1,2}/\d{2,4}\b"), "[DATA]"),
    # Datas escritas sem separador, como "07022022". O dia e o mês precisam ser
    # válidos para não apagar por engano valores de exames.
    (
        re.compile(r"\b(?:0[1-9]|[12]\d|3[01])(?:0[1-9]|1[0-2])(?:19|20)\d{2}\b"),
        "[DATA]",
    ),
    (re.compile(r"\b\d{5}-\d{3}\b"), "[CEP]"),
    # "me chamo joao": o nome pode vir em minúsculas, mas só a primeira palavra
    # é consumida, para não apagar o resto da frase junto.
    (
        re.compile(
            r"(?i:me chamo|meu nome é|chamo-me)"
            r"\s+\w+(?:\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇ]\w+)*"
        ),
        "[NOME]",
    ),
    # "sou o João Silva": aqui a palavra seguinte precisa começar em maiúscula,
    # senão frases como "sou a melhor pessoa para responder" seriam apagadas.
    (
        re.compile(
            r"(?i:sou o|sou a)\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇ]\w+"
            r"(?:\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇ]\w+)*"
        ),
        "[NOME]",
    ),
]


def anonymize_text(text):
    for pattern, replacement in ANONYMIZATION_RULES:
        text = pattern.sub(replacement, text)
    return text


data_cardio["question"] = data_cardio["question"].map(anonymize_text)
data_cardio["answer"] = data_cardio["answer"].map(anonymize_text)

print("Anonimização aplicada em", len(data_cardio), "linhas")

In [ ]:
# 4. Curadoria: descarta vazios, respostas curtas demais e perguntas repetidas.
# O MedPT traz a mesma pergunta respondida por médicos diferentes, então
# mantemos apenas a resposta mais completa de cada pergunta.
data_cardio = data_cardio.dropna(subset=["question", "answer"])
data_cardio["question"] = data_cardio["question"].str.strip()
data_cardio["answer"] = data_cardio["answer"].str.strip()

data_cardio = data_cardio[data_cardio["answer"].str.len() >= MINIMUM_ANSWER_LENGTH]
data_cardio = data_cardio.sort_values(
    "answer", key=lambda coluna: coluna.str.len(), ascending=False
)
data_cardio = data_cardio.drop_duplicates(subset=["question"], keep="first")
data_cardio = data_cardio.reset_index(drop=True)

print("Linhas após a curadoria:", len(data_cardio))

In [ ]:
# 5. Ajuste de registro. O MedPT é um corpus de paciente perguntando e médico
# respondendo, então as respostas saúdam o leitor e mandam "procurar um médico".
# Num assistente usado por quem atende, isso é o registro errado: a saudação é
# removida e as respostas que orientam a buscar atendimento são descartadas.
SAUDACAO_INICIAL = re.compile(
    r"^\s*(ol[áa]|oi|bom dia|boa tarde|boa noite|prezad[oa]s?)\b[\s,.!:;-]*",
    flags=re.IGNORECASE,
)

ORIENTACAO_AO_PACIENTE = re.compile(
    r"\b(procure|consulte|busque|marque|agende|entre em contato com)\b[^.]{0,40}"
    r"\b(m[ée]dico|cardiologista|especialista|profissional|emerg[êe]ncia|"
    r"pronto[- ]socorro|hospital)\b"
    r"|\bseu m[ée]dico\b|\bseu cardiologista\b",
    flags=re.IGNORECASE,
)

data_cardio["answer"] = data_cardio["answer"].map(
    lambda texto: SAUDACAO_INICIAL.sub("", texto, count=1).lstrip()
)
data_cardio = data_cardio[~data_cardio["answer"].str.contains(ORIENTACAO_AO_PACIENTE)]
data_cardio = data_cardio.reset_index(drop=True)

print("Linhas após o ajuste de registro:", len(data_cardio))


In [ ]:
# 6. Montagem das colunas que o restante do notebook espera. O MedPT não tem a
# coluna "Instrução"; ela é derivada de "question_type", que já vem classificada
# no próprio dataset.
# As instruções falam com um profissional de saúde. O MedPT é um corpus de
# paciente perguntando e médico respondendo, então sem esse enquadramento o
# modelo aprende a mandar o leitor "procurar um médico" — o oposto do que este
# assistente precisa fazer.
INSTRUCTION_BY_QUESTION_TYPE = {
    "Tratamento": "Descreva as opções de tratamento indicadas para o quadro a seguir.",
    "Diagnóstico": (
        "Descreva os elementos relevantes para o diagnóstico do quadro a seguir."
    ),
    "Epidemiologia": (
        "Descreva os dados epidemiológicos relevantes para o quadro a seguir."
    ),
    "Estilo de vida saudável": (
        "Descreva as orientações de estilo de vida indicadas para o quadro a seguir."
    ),
    "Anatomia e fisiologia": (
        "Descreva os aspectos anatômicos e fisiológicos relevantes ao quadro a seguir."
    ),
    "Escolha de profissionais de saúde": (
        "Descreva a especialidade e o encaminhamento indicados para o quadro a seguir."
    ),
    "Outros": "Responda de forma técnica à questão clínica a seguir.",
}

DEFAULT_INSTRUCTION = "Responda de forma técnica à questão clínica a seguir."

data = pd.DataFrame(
    {
        "Instrução": data_cardio["question_type"].map(
            lambda tipo: INSTRUCTION_BY_QUESTION_TYPE.get(tipo, DEFAULT_INSTRUCTION)
        ),
        "Pergunta": data_cardio["question"],
        "Resposta": data_cardio["answer"],
    }
)

print("Exemplos prontos para o treino:", len(data))
data.head()

In [ ]:
def format_dataset_into_model_input(data):
    formatted_data = {
        "instruction": data["Instrução"].tolist(),
        "input": data["Pergunta"].tolist(),
        "output": data["Resposta"].tolist()
    }

    with open(OUTPUT_PATH_DATASET, "w", encoding="utf-8") as output_file:
        json.dump(formatted_data, output_file, indent=4, ensure_ascii=False)

    print(f"Dataset salvo em {OUTPUT_PATH_DATASET}")

In [ ]:
format_dataset_into_model_input(data)

In [ ]:

# format_dataset_into_model_input(data)


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",

    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
# alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
alpaca_prompt = """Você é um assistente médico especializado em apoiar profissionais de saúde na area médica de cardiologia.
                    Sua função é responder perguntas médicas utilizando prioritariamente o contexto
                    fornecido pelo sistema de recuperação de documentos.

                    Quem lê a resposta é um profissional de saúde conduzindo o caso, e não o paciente.
                    Não oriente o leitor a procurar um médico ou um serviço de emergência: é ele quem
                    presta o atendimento. Não use saudações e refira-se ao paciente na terceira pessoa.

                    Regras obrigatórias:
                  - Use somente as informações presentes no contexto recuperado.
                  - Se o contexto não for suficiente, declare explicitamente essa limitação.
                  - Não invente dados do paciente, diagnósticos, exames ou fontes.
                  - Não prescreva, inicie, suspenda ou altere medicamentos.
                  - Destaque sinais de alerta e necessidade de avaliação imediata quando aplicável.
                  - Diferencie dados fornecidos, informações das fontes e limitações da análise.
                  - Cite as fontes usando o formato [Fonte: nome do arquivo].
                  - Finalize lembrando que a resposta exige validação de um profissional de saúde.

                  Responda em português do Brasil, de forma clara, objetiva e prudente.

### Instrução:
{}

### Pergunta:
{}

### Resposta:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):

        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

from datasets import load_dataset

OUTPUT_PATH_DATASET = "/content/drive/MyDrive/FIAP/ChallengeFase3/dataset_cardiologiaV2.json"

dataset = load_dataset("json", data_files=OUTPUT_PATH_DATASET, split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

## Separação de treino e teste para não fazer com um prompt manual

In [ ]:
# A divisão usa uma semente fixa, então ela é sempre a mesma - dá para treinar
# hoje e avaliar amanhã, em outra sessão do Colab, com os mesmos exemplos.
splits = dataset.train_test_split(test_size=VALIDATION_FRACTION, seed=SPLIT_SEED)

dataset_treino = splits["train"]
dataset_validacao = splits["test"]

print("Exemplos de treino   :", len(dataset_treino))
print("Exemplos de validação:", len(dataset_validacao))

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset_treino,
    eval_dataset=dataset_validacao,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=max_seq_length,
        dataset_num_proc=2,
        packing=False,

        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,

        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),

        logging_steps=1,

        # Mede a loss nos exemplos de validação de tempos em tempos, para
        # dar com o que comparar a loss de treino.
        eval_strategy="steps",
        eval_steps=EVALUATION_STEPS,
        per_device_eval_batch_size=2,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

In [ ]:
trainer_stats = trainer.train()

## As duas curvas de loss

O `trainer` guarda o histórico em `trainer.state.log_history`. O gráfico abaixo
põe as duas curvas no mesmo eixo:

- as duas caindo juntas: o modelo está aprendendo;
- treino caindo e validação subindo: ele começou a decorar (*overfitting*), e é
  hora de parar antes.

In [ ]:
import matplotlib.pyplot as plt

TRAIN_COLOR = "#2a78d6"
VALIDATION_COLOR = "#eb6834"
SURFACE_COLOR = "#fcfcfb"
TEXT_PRIMARY = "#0b0b0b"
TEXT_SECONDARY = "#52514e"
GRID_COLOR = "#e5e4e0"

# O log_history mistura registros de treino ("loss"), de validação
# ("eval_loss") e o resumo final, que não tem nenhuma das duas.
historico = trainer.state.log_history
loss_treino = [
    (r["step"], r["loss"]) for r in historico if "loss" in r and "step" in r
]
loss_validacao = [
    (r["step"], r["eval_loss"]) for r in historico if "eval_loss" in r and "step" in r
]

figura, eixo = plt.subplots(figsize=(8, 4.5), dpi=150)
figura.patch.set_facecolor(SURFACE_COLOR)
eixo.set_facecolor(SURFACE_COLOR)

ultimo_passo = 0
for pontos, rotulo, cor, marcador in (
    (loss_treino, "Treino", TRAIN_COLOR, None),
    (loss_validacao, "Validação", VALIDATION_COLOR, "o"),
):
    if not pontos:
        continue

    passos = [passo for passo, _ in pontos]
    perdas = [perda for _, perda in pontos]
    ultimo_passo = max(ultimo_passo, passos[-1])

    eixo.plot(
        passos, perdas, color=cor, linewidth=2,
        label=rotulo, marker=marcador, markersize=5,
    )
    # Rótulo direto no fim da linha: a cor identifica a série, o texto usa a
    # tinta padrão para continuar legível em impressão e em daltonismo.
    eixo.annotate(
        rotulo, xy=(passos[-1], perdas[-1]), xytext=(8, 0),
        textcoords="offset points", color=TEXT_PRIMARY, fontsize=9, va="center",
    )

eixo.set_xlim(left=0, right=ultimo_passo * 1.15)
eixo.set_title(
    "Loss de treino x loss de validação",
    color=TEXT_PRIMARY, fontsize=12, pad=12, loc="left",
)
eixo.set_xlabel("Passo do treino", color=TEXT_SECONDARY, fontsize=9)
eixo.set_ylabel("Loss", color=TEXT_SECONDARY, fontsize=9)

eixo.grid(axis="y", color=GRID_COLOR, linewidth=0.8)
eixo.set_axisbelow(True)
for lado in ("top", "right"):
    eixo.spines[lado].set_visible(False)
for lado in ("left", "bottom"):
    eixo.spines[lado].set_color(GRID_COLOR)
eixo.tick_params(colors=TEXT_SECONDARY, labelsize=8)

legenda = eixo.legend(frameon=False, loc="upper right", fontsize=9)
for texto in legenda.get_texts():
    texto.set_color(TEXT_PRIMARY)

figura.tight_layout()
figura.savefig(
    "/content/drive/MyDrive/FIAP/ChallengeFase3/curva_de_loss.png",
    facecolor=SURFACE_COLOR, bbox_inches="tight",
)
plt.show()

In [ ]:

FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Responda esta pergunta que solicita informação",
        "Quais são os sinais de alerta de um infarto agudo do miocárdio?", # input
        "",
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)

input_length = inputs["input_ids"].shape[1]

generated_tokens = outputs[0][input_length:]

tokenizer.batch_decode(generated_tokens)

# tokenizer.batch_decode(outputs)

In [ ]:

FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Responda esta pergunta que solicita informação",
        "Quais são os sinais de alerta de um infarto agudo do miocárdio?", # input
        "",
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

In [ ]:
model.save_pretrained("/content/drive/MyDrive/FIAP/ChallengeFase3/lora_model") # Local saving


In [ ]:
tokenizer.save_pretrained("/content/drive/MyDrive/FIAP/ChallengeFase3/lora_model")


In [ ]:
model.save_pretrained_merged(
    "/content/drive/MyDrive/FIAP/ChallengeFase3/medical_model",
    tokenizer,
    save_method="merged_16bit"
)